# Primal–Dual Algorithm for Minimum Weight Perfect Matching
## Bipartite Graph (Augmenting Path Method)

### Assignment Submission

Group Members:  
- Piyush Anand — CS25MTECH12009  
- Darshanraj Pattanaik — CS25MTECH12002  

---

## Objective

Implement the Primal–Dual (augmenting path) algorithm to compute a Minimum Weight Perfect Matching in a bipartite graph.

The task is to:

1. Read a bipartite weighted graph from a 3-row CSV.
2. Construct the cost matrix.
3. Run the primal–dual matching algorithm.
4. Print dual variables u and v after every iteration.
5. Print the final matching and its cost.

---

## Input Format (CSV)

Each CSV contains exactly 3 rows:

Row 1: Left-side vertex IDs  
Row 2: Right-side vertex IDs  
Row 3: Edge weights  

Each column represents one edge of the bipartite graph.

Example:

u | v | w  
1 | 4 | 8  
1 | 5 | 2  
2 | 4 | 7  

---

## Output Format

For each iteration of the algorithm, print:

Iteration k  
u = [...]  
v = [...]  
Current matching = [...]  

Finally print:

Final Matching  
Total Minimum Cost  
Final Dual Variables  

---

## Algorithm Description

We implement the primal–dual augmenting path algorithm, often used to compute a minimum weight perfect matching in bipartite graphs.

Key ideas:

1. Maintain dual variables for left set (u) and right set (v).
2. Maintain reduced costs.
3. Use an augmenting path mechanism to adjust matchings.
4. Adjust dual variables to maintain feasibility.
5. Iteratively improve the matching until all left vertices are matched.


---


## Generated Testcases for Testing

In [ ]:


import pandas as pd
testcases = {
1: ([1,1,1,2,2,3,3,3],
    [101,102,103,101,103,101,102,103],
    [4,1,3,2,8,7,6,9]),

2: ([1,1,2,2,3,3],
    [101,102,102,103,101,103],
    [5,7,3,4,2,8]),

3: ([1,1,1,2,2,3,3],
    [101,102,103,102,103,101,103],
    [9,1,6,2,8,7,3]),

4: ([1,1,2,2,3,3],
    [101,102,101,103,102,103],
    [10,5,2,3,8,6]),

5: ([1,1,1,2,2,2,3,3,3],
    [101,102,103,101,102,103,101,102,103],
    [2,9,5,3,4,8,6,7,1]),

6: ([1,1,2,2,3,3,4,4],
    [101,102,103,104,101,103,102,104],
    [8,6,2,7,1,9,3,5]),

7: ([1,1,1,2,2,3,3],
    [101,102,103,101,103,101,102],
    [3,8,6,4,2,5,7]),

8: ([1,1,2,2,3,3],
    [101,103,101,102,102,103],
    [2,5,6,3,4,1]),

9: ([1,1,2,2,3,3,4,4],
    [101,102,101,103,102,104,103,104],
    [1,9,3,6,8,2,4,7]),

10: ([1,1,2,2,3,3],
     [101,102,101,103,102,103],
     [7,2,3,4,5,6]),

11: ([1,1,2,2,3,3,4,4],
     [101,102,103,104,101,102,103,104],
     [9,1,3,5,2,6,4,7]),

12: ([1,1,2,2,3,3],
     [101,102,103,101,102,103],
     [5,3,9,4,6,2]),

13: ([1,1,1,2,2,3,3],
     [101,102,103,101,103,102,103],
     [2,4,6,3,5,7,1]),

14: ([1,1,2,2,3,3,4,4],
     [101,102,103,104,101,103,102,104],
     [2,3,6,8,4,9,5,7]),

15: ([1,1,2,2,3,3,4,4],
     [101,102,101,103,102,104,103,104],
     [9,4,2,6,5,3,7,8]),
}

for i, (U, V, W) in testcases.items():
    df = pd.DataFrame([U, V, W])
    fname = f"testcase_{i}.csv"
    df.to_csv(fname, header=False, index=False)
    print(f" Saved {fname} (edges = {len(U)})")

print("\nAll 15 testcases created successfully in the current directory!")


 Saved testcase_1.csv (edges = 8)
 Saved testcase_2.csv (edges = 6)
 Saved testcase_3.csv (edges = 7)
 Saved testcase_4.csv (edges = 6)
 Saved testcase_5.csv (edges = 9)
 Saved testcase_6.csv (edges = 8)
 Saved testcase_7.csv (edges = 7)
 Saved testcase_8.csv (edges = 6)
 Saved testcase_9.csv (edges = 8)
 Saved testcase_10.csv (edges = 6)
 Saved testcase_11.csv (edges = 8)
 Saved testcase_12.csv (edges = 6)
 Saved testcase_13.csv (edges = 7)
 Saved testcase_14.csv (edges = 8)
 Saved testcase_15.csv (edges = 8)

All 15 testcases created successfully in the current directory!


In [ ]:
import numpy as np
import pandas as pd

def read_csv_to_cost(filename):
    df = pd.read_csv(filename, header=None)

    U = df.iloc[0].astype(int).to_numpy()
    V = df.iloc[1].astype(int).to_numpy()
    W = df.iloc[2].astype(float).to_numpy()

    left_nodes = np.unique(U)
    right_nodes = np.unique(V)

    nL = len(left_nodes)
    nR = len(right_nodes)

    if nL != nR:
        print(f"\n Matching not possible in {filename}: Left({nL}) != Right({nR})")
        return None, None, None, None, False

    n = nL

    idxL = {left_nodes[i]: i for i in range(n)}
    idxR = {right_nodes[j]: j for j in range(n)}

    INF = 10**9
    cost = np.full((n, n), INF)

    for u, v, w in zip(U, V, W):
        cost[idxL[u], idxR[v]] = w

    for i in range(n):
        if np.all(cost[i] >= INF):
            print(f"\n Matching not possible in {filename}: Left node {left_nodes[i]} has no edges.")
            return None, None, None, None, False

    return cost, left_nodes, right_nodes, n, True


def min_cost_matching(cost):
    n = cost.shape[0]
    INF = 10**15

    matchL = [-1] * n
    matchR = [-1] * n

    u = [0.0] * n
    v = [0.0] * n

    # Initialize duals
    for i in range(n):
        row_min = float(min(cost[i]))
        if row_min >= 10**9:
            return None, None, u, v, False  # last duals
        u[i] = row_min

    print("Initial dual variables:")
    print("u =", [int(x) for x in u])
    print("v =", [int(x) for x in v])

    for iteration in range(n):

        L = iteration

        dist = [INF] * n
        parent = [-1] * n
        used = [False] * n

        for j in range(n):
            dist[j] = cost[L][j] - u[L] - v[j]

        while True:
            j_min = -1
            best = INF

            for j in range(n):
                if not used[j] and dist[j] < best:
                    best = dist[j]
                    j_min = j

            if j_min == -1:
                return None, None, u, v, False  # return last duals

            used[j_min] = True

            if matchR[j_min] == -1:
                break

            i2 = matchR[j_min]
            for j2 in range(n):
                if not used[j2]:
                    newDist = dist[j_min] + (cost[i2][j2] - u[i2] - v[j2])
                    if newDist < dist[j2]:
                        dist[j2] = newDist
                        parent[j2] = j_min

        delta = dist[j_min]

        for j in range(n):
            if used[j]:
                v[j] += delta - dist[j]
                i2 = matchR[j]
                if i2 != -1:
                    u[i2] -= delta - dist[j]

        u[L] += delta

        print("\nIteration", iteration + 1)
        print("Updated dual variables:")
        print("u =", [round(float(x), 4) for x in u])
        print("v =", [round(float(x), 4) for x in v])

        cur = j_min
        while parent[cur] != -1:
            prev = parent[cur]
            i2 = matchR[prev]
            matchR[cur] = matchR[prev]
            matchL[i2] = cur
            cur = prev

        matchL[L] = cur
        matchR[cur] = L

        print("Current matching:", matchL)

    total_cost = sum(cost[i][matchL[i]] for i in range(n))
    return matchL, total_cost, u, v, True


def run(filename):
    cost, left, right, n, ok = read_csv_to_cost(filename)

    if not ok:
        return

    print("\nRunning Min-Cost Matching for", filename)

    match, total_cost, u, v, ok2 = min_cost_matching(cost)

    if not ok2:
        print(f"\nMatching not possible in {filename}: Graph is not fully matchable.")
        print("\nLast computed dual variables:")
        print("u =", [round(float(x), 4) for x in u])
        print("v =", [round(float(x), 4) for x in v])
        return

    print("\n=== Final Matching ===")
    for i in range(n):
        print(f"{left[i]} → {right[match[i]]}  (cost = {cost[i][match[i]]})")

    print("Total minimum cost =", total_cost)
    print("Final duals:")
    print("u =", [round(float(x), 4) for x in u])
    print("v =", [round(float(x), 4) for x in v])


if __name__ == "__main__":
    for t in range(1, 21):
        run(f"testcase_{t}.csv")



Running Min-Cost Matching for testcase_1.csv
Initial dual variables:
u = [1, 2, 6]
v = [0, 0, 0]

Iteration 1
Updated dual variables:
u = [1.0, 2.0, 6.0]
v = [0.0, 0.0, 0.0]
Current matching: [1, -1, -1]

Iteration 2
Updated dual variables:
u = [1.0, 2.0, 6.0]
v = [0.0, 0.0, 0.0]
Current matching: [1, 0, -1]

Iteration 3
Updated dual variables:
u = [-1.0, 1.0, 8.0]
v = [1.0, 2.0, 0.0]
Current matching: [2, 0, 1]

=== Final Matching ===
1 → 103  (cost = 3)
2 → 101  (cost = 2)
3 → 102  (cost = 6)
Total minimum cost = 11
Final duals:
u = [-1.0, 1.0, 8.0]
v = [1.0, 2.0, 0.0]

Running Min-Cost Matching for testcase_2.csv
Initial dual variables:
u = [5, 3, 2]
v = [0, 0, 0]

Iteration 1
Updated dual variables:
u = [5.0, 3.0, 2.0]
v = [0.0, 0.0, 0.0]
Current matching: [0, -1, -1]

Iteration 2
Updated dual variables:
u = [5.0, 3.0, 2.0]
v = [0.0, 0.0, 0.0]
Current matching: [0, 1, -1]

Iteration 3
Updated dual variables:
u = [2.0, 2.0, 5.0]
v = [3.0, 1.0, 0.0]
Current matching: [1, 2, 0]

=== 